In [1]:
import cv2

# 1. Initialize the capture object. 
# Pass 0 for the default built-in webcam. Pass a file path string to read a video file.
cap = cv2.VideoCapture(0)

# Check if the camera/file opened successfully
if not cap.isOpened():
    print("Error: Could not open video source.")
    exit()

while True:
    # 2. Capture frame-by-frame
    # ret is a boolean (True if frame read successfully), frame is the image array
    ret, frame = cap.read()
    
    # If the frame wasn't grabbed successfully (e.g. end of video file), break the loop
    if not ret:
        print("Error: Can't receive frame.")
        break

    # 3. Display the resulting frame
    cv2.imshow('Webcam Feed', frame)

    # 4. Break the loop if 'q' key is pressed
    # cv2.waitKey(1) returns the code of the pressed key every 1 millisecondq
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# 5. When everything is done, release the capture and destroy windows
cap.release()
cv2.destroyAllWindows()

In [5]:
import cv2

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Error: Could not open video source.")
    exit()

# Learns what the "empty room" looks like and flags anything that changes
subtractor = cv2.createBackgroundSubtractorMOG2(
    history=500, varThreshold=50, detectShadows=True
)
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
MIN_AREA = 1500  # ignore blobs smaller than this (noise)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.resize(frame, (640, 480))

    # Image operations: grayscale + blur to reduce noise
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (21, 21), 0)

    # Segmentation: background subtraction, then clean the mask
    mask = subtractor.apply(blur)
    _, mask = cv2.threshold(mask, 200, 255, cv2.THRESH_BINARY)  # drops shadows
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.dilate(mask, kernel, iterations=2)

    # Analysis: find moving blobs and draw boxes
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    motion = False
    for c in contours:
        if cv2.contourArea(c) < MIN_AREA:
            continue
        motion = True
        x, y, w, h = cv2.boundingRect(c)
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

    status = "MOTION DETECTED" if motion else "Area clear"
    color = (0, 0, 255) if motion else (0, 255, 0)
    cv2.putText(frame, status, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

    cv2.imshow("Intruder Alarm", frame)
    cv2.imshow("Mask", mask)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [8]:
import cv2, os

name = input("Enter person's name: ").strip()
folder = os.path.join("faces", name)
os.makedirs(folder, exist_ok=True)

detector = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)
cap = cv2.VideoCapture(0)
count, TARGET = 0, 40

while count < TARGET:
    ret, frame = cap.read()
    if not ret:
        break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    if detector.empty():
        raise RuntimeError(
            "Failed to load face cascade: "
            + (cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
        )

    faces = detector.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(100, 100),
        flags=cv2.CASCADE_SCALE_IMAGE
    )

    for (x, y, w, h) in faces[:1]:  # one face at a time
        crop = cv2.resize(gray[y:y + h, x:x + w], (200, 200))
        cv2.imwrite(os.path.join(folder, f"{count}.jpg"), crop)
        count += 1
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

    cv2.putText(frame, f"Captured {count}/{TARGET}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    cv2.imshow("Register", frame)
    if cv2.waitKey(150) & 0xFF == ord('q'):  # delay helps vary your pose
        break

cap.release()
cv2.destroyAllWindows()

RuntimeError: Failed to load face cascade: C:\Users\User\AppData\Roaming\Python\Python314\site-packages\cv2\data\haarcascade_frontalface_default.xml

In [6]:
detector = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)